# Project 10: welk deel van de wegen in Paramaribo is verhard?

**De vraag.** Welk deel van de wegsegmenten in het district Paramaribo is verhard? Als een onderzoeksteam alleen een willekeurige steekproef van segmenten controleerde, hoe dicht zou zijn schatting bij de volledige telling liggen, en hoe breed moet het interval zijn dat het rapporteert?

**De data.** Elk wegsegment in het district Paramaribo in OpenStreetMap, gedownload via de Overpass API op 2026-11-06. Kaartdata © OpenStreetMap-bijdragers, onder de Open Database License. OpenStreetMap legt een wegdek alleen vast waar een kaartmaker dat heeft toegevoegd, dus in dit project betekent "aandeel verhard" het aandeel onder de segmenten met een surface-tag.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week is de datacontroleur verantwoordelijk voor de vraag welke segmenten het antwoord beschrijft.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, één grafiek van schattingen en intervallen, een notitie van ongeveer 200 woorden voor de wegplanners van het ministerie dat over openbare werken gaat, jullie AI-logboek, en één extra interval uit een steekproef van 100 segmenten, getrokken met jullie teamnummer als seed.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of downloadt hem via de Overpass API als het bestand er niet is. Elke rij is één wegsegment: een stuk weg tussen twee punten waar een kaartmaker de weg heeft opgeknipt. De kolommen zijn het OpenStreetMap-id, de wegklasse (`highway`), de `surface`-tag, `paved` (1 voor verhard, 0 voor onverhard, leeg als er geen surface-tag is) en de lengte van het segment in meters.

In [ ]:
import json, math, urllib.parse, urllib.request
import numpy as np
import pandas as pd

QUERY = ('[out:json][timeout:180];'
         'area["ISO3166-2"="SR-PM"]->.paramaribo;'
         'way["highway"~"^(primary|secondary|tertiary|unclassified|residential|living_street|primary_link|secondary_link|tertiary_link)$"](area.paramaribo);'
         'out tags geom;')
# de groepen verhard en onverhard van de surface-tag, uit de OpenStreetMap-wiki (Key:surface)
PAVED = ["paved", "asphalt", "chipseal", "concrete", "concrete:lanes", "concrete:plates", "paving_stones",
         "paving_stones:lanes", "grass_paver", "sett", "unhewn_cobblestone", "cobblestone", "bricks", "metal",
         "metal_grid", "wood", "stepping_stones", "tiles", "fibre_reinforced_polymer_grate", "resin_bound"]
UNPAVED = ["unpaved", "compacted", "fine_gravel", "gravel", "shells", "rock", "pebblestone", "ground", "dirt",
           "earth", "mud", "laterite", "grass", "sand", "woodchips", "snow", "ice", "salt", "clay"]


def metres(points):
    # lengte langs de punten van een segment, op een bol met een straal van 6.371 km
    d = 0.0
    for a, b in zip(points, points[1:]):
        p1, p2, dl = math.radians(a["lat"]), math.radians(b["lat"]), math.radians(b["lon"] - a["lon"])
        h = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
        d += 2 * 6371000 * math.asin(math.sqrt(h))
    return d


try:
    roads = pd.read_csv("../data/week10-paramaribo-roads.csv")
except FileNotFoundError:
    # de live kaart verandert als mensen haar bewerken, dus je getallen kunnen een beetje afwijken van het cursusbestand
    req = urllib.request.Request("https://overpass-api.de/api/interpreter",
                                 data=urllib.parse.urlencode({"data": QUERY}).encode(),
                                 headers={"User-Agent": "BiGiData-course/1.0"})
    ways = json.load(urllib.request.urlopen(req, timeout=240))["elements"]
    rows = []
    for way in ways:
        s = way["tags"].get("surface")
        rows.append({"osm_id": way["id"], "highway": way["tags"]["highway"], "surface": s,
                     "paved": 1.0 if s in PAVED else 0.0 if s in UNPAVED else np.nan,
                     "length_m": round(metres(way["geometry"]), 1)})
    roads = pd.DataFrame(rows)
roads

## TODO 1: beschrijf de tabel

Druk het aantal segmenten af en het aantal per wegklasse. Sla in `n_tagged` het aantal segmenten op dat een surface-tag heeft.

In [ ]:
# TODO 1
n_tagged = None

In [ ]:
# CHECK 1
assert n_tagged is not None, "n_tagged is nog leeg"
assert n_tagged == roads["surface"].notna().sum()
print("CHECK 1 geslaagd:", n_tagged, "van de", len(roads), "segmenten hebben een surface-tag")

## TODO 2: wie kreeg een tag?

Sla in `coverage` per wegklasse het aandeel segmenten met een surface-tag op, als een pandas Series met de klasse als index. Druk daarna per klasse het aandeel verhard af onder de segmenten met een tag.

Beantwoord in een tekstcel drie vragen. Welke klasse met minstens 100 segmenten heeft het minst vaak een tag? Zijn de segmenten met een tag in die klasse vaker of minder vaak verhard dan de rest? Wat zegt dat over de segmenten zonder tag?

In [ ]:
# TODO 2
coverage = None

In [ ]:
# CHECK 2
expected = roads["surface"].notna().groupby(roads["highway"]).mean()
assert coverage is not None, "coverage is nog leeg"
assert set(coverage.index) == set(expected.index)
assert all(abs(coverage[k] - expected[k]) < 1e-9 for k in expected.index)
big = roads["highway"].value_counts()
print("CHECK 2 geslaagd:", {k: round(float(expected[k]), 3) for k in big[big >= 100].index})

## TODO 3: de volledige telling

Vanaf hier zijn de segmenten met een tag je populatie. Sla ze op in een tabel met de naam `tagged`, en sla hun aandeel verhard op in `true_share`. In echt onderzoekswerk kent niemand dit getal. Hier ken je het wel, dus je kunt elke steekproef eraan meten.

In [ ]:
# TODO 3
tagged = None
true_share = None

In [ ]:
# CHECK 3
assert tagged is not None and true_share is not None, "tagged of true_share is nog leeg"
assert len(tagged) == roads["paved"].notna().sum()
assert abs(true_share - roads["paved"].mean()) < 1e-9
print("CHECK 3 geslaagd: de volledige telling omvat", len(tagged), "segmenten met een tag")

## TODO 4: één onderzoek van 100 segmenten

Trek een willekeurige steekproef van 100 segmenten met een tag met `tagged.sample(n=100, random_state=10)` en sla die op in `sample_100`. Sla het aandeel verhard daarin op in `est_100`.

Bouw daarna een bootstrap-95%-interval: maak een generator met `np.random.default_rng(10)`, trek 2.000 hersteekproeven van de 100 waarden van `paved` met teruglegging, en sla het 2,5e en het 97,5e percentiel van hun aandelen op in `lo_100` en `hi_100`. Druk af of het interval `true_share` bevat.

In [ ]:
# TODO 4
sample_100 = None
est_100 = None
lo_100, hi_100 = None, None

In [ ]:
# CHECK 4
assert sample_100 is not None and est_100 is not None and lo_100 is not None, "een variabele van TODO 4 is nog leeg"
assert len(sample_100) == 100 and sample_100["osm_id"].is_unique
assert set(sample_100["osm_id"]) == set(tagged.sample(n=100, random_state=10)["osm_id"]), "trek de steekproef met random_state=10"
assert abs(est_100 - sample_100["paved"].mean()) < 1e-9
se_100 = np.sqrt(est_100 * (1 - est_100) / 100)
assert lo_100 < est_100 < hi_100, "het interval moet de schatting bevatten"
assert 0.6 * 2 * 1.96 * se_100 < hi_100 - lo_100 < 1.6 * 2 * 1.96 * se_100, "de breedte ligt ver van wat de standaardfout voorspelt"
print("CHECK 4 geslaagd: breedte van het interval", round(hi_100 - lo_100, 3))

## TODO 5: steekproeven van verschillende grootte

Herhaal TODO 4 voor steekproeven van 25, 50, 100, 200 en 400 segmenten. Trek elke steekproef met `random_state` gelijk aan haar grootte, zodat de steekproeven los van elkaar staan, en bootstrap elke steekproef met een nieuwe `np.random.default_rng(10)`. Sla in `results` een tabel op met één rij per grootte en de kolommen `n`, `estimate`, `lo`, `hi`, `width` en `holds_full_count`.

Maak één grafiek: de schatting voor elke grootte met het interval als foutbalk, en de volledige telling als stippellijn. Geef de grafiek een titel en benoem beide assen.

In [ ]:
# TODO 5
results = None

In [ ]:
# CHECK 5
assert results is not None, "results is nog leeg"
assert list(results["n"]) == [25, 50, 100, 200, 400]
assert all(results["lo"] <= results["estimate"]) and all(results["estimate"] <= results["hi"])
assert results["width"].iloc[-1] < results["width"].iloc[:3].min(), "het interval hoort smaller te worden als de steekproef groeit"
print("CHECK 5 geslaagd: breedtes", [round(v, 3) for v in results["width"]])

## TODO 6: klopt 95 procent?

Eén interval bevat de volledige telling of mist haar. Om de methode te testen, herhaal je het onderzoek 200 keer. Maak een generator met `np.random.default_rng(10)`; trek bij elke herhaling 100 segmenten zonder teruglegging uit de waarden van `paved` in `tagged`, bouw een bootstrap-interval uit 1.000 hersteekproeven, en noteer of het `true_share` bevat. Sla het aandeel van de 200 intervallen dat haar bevat op in `coverage_rate`.

In [ ]:
# TODO 6
coverage_rate = None

In [ ]:
# CHECK 6
assert coverage_rate is not None, "coverage_rate is nog leeg"
assert 0.85 <= coverage_rate <= 1.0, "een 95%-methode hoort de volledige telling in de meeste van de 200 onderzoeken te bevatten"
print("CHECK 6 geslaagd: dekking", round(coverage_rate, 3))

## TODO 7: de AI-controle

Vraag je AI-assistent eerst, zonder hem data te geven: *"Welk deel van de wegen in Paramaribo is verhard?"* Schrijf het antwoord in jullie AI-logboek. Gaf de assistent één getal of een bereik? Noemde hij een bron en een datum?

Druk daarna `list(sample_100["paved"].astype(int))` af, plak de 100 waarden in de chat en vraag: *"Dit zijn 100 willekeurig gekozen wegsegmenten, 1 voor verhard en 0 voor onverhard. Geef me een 95%-interval voor het aandeel verhard onder alle segmenten."* Noteer het interval en de formule die de assistent gebruikte.

Sla in de cel hieronder twee intervallen op als paren (laag, hoog). Gebruik in `sd_interval` `est_100` plus en min 1,96 keer de standaardafwijking van de 100 waarden. Gebruik in `se_interval` in plaats daarvan 1,96 keer de standaardfout. Druk beide af naast je bootstrap-interval, en schrijf in je logboek op welk van de twee op het antwoord van de AI lijkt.

In [ ]:
# TODO 7
sd_interval = None
se_interval = None

In [ ]:
# CHECK 7
assert sd_interval is not None and se_interval is not None, "sd_interval of se_interval is nog leeg"
sd_width = sd_interval[1] - sd_interval[0]
se_width = se_interval[1] - se_interval[0]
assert abs(sd_width / se_width - 10) < 0.01, "de standaardfout is de standaardafwijking gedeeld door de wortel van 100"
assert abs(se_width - (hi_100 - lo_100)) < 0.3 * (hi_100 - lo_100), "het interval met de standaardfout hoort dicht bij het bootstrap-interval te liggen"
print("CHECK 7 geslaagd: het interval met de standaardafwijking is", round(sd_width / se_width, 1), "keer zo breed als het interval met de standaardfout")

## TODO 8: segmenten of kilometers?

Een segment is een eenheid van de kaart, niet van het wegennet: een lange laan kan tien segmenten zijn en een zijstraat één. Een wegplanner denkt in kilometers. Sla in `length_share` het aandeel van de lengte met een tag op dat verhard is, en vergelijk het met `true_share`. Zeg in een tekstcel welk van de twee de vraag van de planner beantwoordt, en waarom ze verschillen.

In [ ]:
# TODO 8
length_share = None

In [ ]:
# CHECK 8
assert length_share is not None, "length_share is nog leeg"
expected = (tagged["paved"] * tagged["length_m"]).sum() / tagged["length_m"].sum()
assert abs(length_share - expected) < 1e-6
print("CHECK 8 geslaagd: het aandeel naar lengte verschilt van het aandeel naar segmenten met", round(abs(length_share - true_share), 3))

## TODO 9: de notitie voor de wegplanners

Schrijf in een tekstcel hieronder ongeveer 200 woorden voor de wegplanners van het ministerie dat over openbare werken gaat. Zij beslissen welke straten worden onderzocht en welke het eerst worden verhard, dus geef hun iets waar een planner mee kan werken.

- De schatting die een onderzoek van 100 segmenten zou hebben opgeleverd, met het interval, in één eerlijke zin.
- De volledige telling, hoeveel segmenten geen surface-tag hebben, en uit welke wegklassen ze komen.
- Hoeveel segmenten een onderzoeksteam zou moeten bezoeken om het aandeel verhard te kennen tot op 5 procentpunten naar boven of beneden, uit je tabel van TODO 5 of met de formule uit het vaardighedennotebook.
- De bron en de datum, en één ding dat het interval de planners niet kan vertellen.